In [6]:
import numpy as np
import pandas as pd
import sklearn.model_selection
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
raw = pd.read_excel('../data/raw/disaster_prediction_dataset.xlsx')
print('Raw shape:', raw.shape)

# Class grouping
TYPE_MAP = {
    'Flood': 'Flood',
    'Glacial lake outburst flood': 'Flood',
    'Mass movement (wet)': 'Landslide',
    'Mass movement (dry)': 'Landslide',
    'Earthquake': 'Earthquake',
    'Impact': 'Earthquake',
    'Epidemic': 'Epidemic',
    'Infestation': 'Epidemic',
    'Animal incident': 'Epidemic',
    'Extreme temperature': 'Extreme Temp',
    'Fog': 'Extreme Temp',
    'Storm': 'Storm',
    'Drought': 'Drought',
    'Wildfire': 'Wildfire',
    # Volcanic activity dropped
}

df = raw.copy()
df['target'] = df['Disaster Type'].map(TYPE_MAP)
df = df[df['target'].notna()].copy()

print('After target mapping:', df.shape)
print(df['target'].value_counts())

Raw shape: (17756, 47)
After target mapping: (17474, 48)
target
Flood           6183
Storm           5053
Earthquake      1651
Epidemic        1622
Landslide        918
Drought          803
Extreme Temp     730
Wildfire         514
Name: count, dtype: int64


In [8]:
cols_to_drop = [
    # Identifiers
    'DisNo.', 'Classification Key', 'External IDs', 'Event Name',
    'ISO', 'Location', 'River Basin', 'Admin Units', 'GADM Admin Units',
    'Entry Date', 'Last Update',

    # Label hierarchy leakage
    'Historic', 'Disaster Group', 'Disaster Subgroup',
    'Disaster Type', 'Disaster Subtype',

    # Post-event impact
    'Total Deaths', 'No. Injured', 'No. Affected', 'No. Homeless', 'Total Affected',
    "Total Damage ('000 US$)", "Total Damage, Adjusted ('000 US$)",
    "Reconstruction Costs ('000 US$)", "Reconstruction Costs, Adjusted ('000 US$)",
    "Insured Damage ('000 US$)", "Insured Damage, Adjusted ('000 US$)",

    # Post-event response
    "AID Contribution ('000 US$)", 'Appeal', 'OFDA/BHA Response', 'Declaration',

    # High missing / less useful
    'Latitude', 'Longitude', 'CPI',

    # Optional: Origin & Associated Types can be borderline.
    # Keep them for now (often known near the time of the event).
    # 'Origin', 'Associated Types',
]

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print('Columns kept:')
print(df.columns.tolist())
print('\nShape:', df.shape)

Columns kept:
['Country', 'Subregion', 'Region', 'Origin', 'Associated Types', 'Magnitude', 'Magnitude Scale', 'Start Year', 'Start Month', 'Start Day', 'End Year', 'End Month', 'End Day', 'target']

Shape: (17474, 14)


In [9]:
train_df, test_df = sklearn.model_selection.train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df['target']
)

print(f'Train: {train_df.shape[0]:,} rows')
print(f'Test : {test_df.shape[0]:,} rows')
print('\nTrain class distribution:')
print(train_df['target'].value_counts(normalize=True).round(3))

Train: 13,979 rows
Test : 3,495 rows

Train class distribution:
target
Flood           0.354
Storm           0.289
Earthquake      0.094
Epidemic        0.093
Landslide       0.053
Drought         0.046
Extreme Temp    0.042
Wildfire        0.029
Name: proportion, dtype: float64


In [10]:
train_df.to_excel('../data/processed/disaster_train.xlsx',  index=False)
test_df.to_excel('../data/processed/disaster_test.xlsx',  index=False)